In [12]:
%pip install requests
%pip install deltalake
%pip install pyarrow
%pip install pandas
%pip install deltalake

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
from deltalake import DeltaTable
# FUCIONES
from utils import upsert_data_as_delta, read_most_recent_partition_data


## GET DATA

In [8]:
## 
from pathlib import Path

bronze_dir = "datalake/bronze/FM_SP_api"
raw_dir = f"/{bronze_dir}"
df_artists = DeltaTable(raw_dir)
df_artists = df_artists.to_pandas()

## Busco los registros que forman parte de la extracción actual.
raw_dir = Path("/datalake/bronze/FM_SP_api_top_100")
df_artists_in_top_100 = read_most_recent_partition_data(raw_dir)
df_artists_in_top_100


,SP_track_id,date_extraction,__index_level_0__
0,48UPSzbZjgc449aqz8bxox,2024-09-12,NaN
1,3AJwUDP919kvQ9QcozQPxg,2024-09-12,NaN
2,0DiWol3AO6WpXZgp0goxAV,2024-09-12,NaN
3,0pQskrTITgmCMyr85tb9qq,2024-09-12,NaN
4,3oqWr0jDWNXxWufNogGREp,2024-09-12,NaN
5,70LcF31zb1H0PyJoS1Sx1r,2024-09-12,0.0
6,7tFiyTwD0nx5a1eklYtX2J,2024-09-12,2.0
7,5GjPQ0eI7AgmOnADn1EO6Q,2024-09-12,4.0
8,2lpIh6Gr6HYjg1CFBaucS5,2024-09-12,5.0
9,6H3kDe7CGoWYBabAeVWGiD,2024-09-12,6.0


## Transformaciones

In [9]:
df_artists['FM_top_track_listeners'] = df_artists['FM_top_track_listeners'].astype(int)
df_artists['FM_top_track_playcount'] = df_artists['FM_top_track_playcount'].astype(int)
df_artists['SP_album_release_date'] = pd.to_datetime(df_artists['SP_album_release_date'])
df_artists['date_extraction'] = pd.to_datetime(df_artists['date_extraction'])



df_artists.dtypes

FM_artist_mbid                    object
FM_artist_name                    object
FM_rank                            int64
FM_top_track_name                 object
FM_top_track_mbid                 object
FM_top_track_listeners             int64
FM_top_track_playcount             int64
SP_track_id                       object
SP_album_release_date     datetime64[ns]
SP_track_popularity                int64
date_extraction           datetime64[us]
__index_level_0__                  int64
dtype: object

## CALCULOS

In [10]:
import numpy as np


spotify_popularity_mean = df_artists['SP_track_popularity'].mean()
df_artists['SP_track_popularity_media_distance'] = df_artists['SP_track_popularity'].apply(lambda x: abs(x - spotify_popularity_mean))
# Calcular la década
df_artists['SP_album_decada'] = df_artists['SP_album_release_date'].apply(lambda x: (x.year // 10) * 10)
# 
df_artists['is_in_top_100'] = df_artists['SP_track_id'].isin(df_artists_in_top_100['SP_track_id'])
df_artists

,FM_artist_mbid,FM_artist_name,FM_rank,FM_top_track_name,FM_top_track_mbid,FM_top_track_listeners,FM_top_track_playcount,SP_track_id,SP_album_release_date,SP_track_popularity,date_extraction,__index_level_0__,SP_track_popularity_media_distance,SP_album_decada,is_in_top_100
0,b10bbbfc-cf9e-42e0-be17-e2c3e1d2600d,The Beatles,5,Eleanor Rigby,424c10aa-a857-4dc6-872e-2a8fb0b707f3,1091344,7357604,5GjPQ0eI7AgmOnADn1EO6Q,1966-08-05,72,2024-09-12,4,4.733333,1960,True
1,b071f9fa-14b0-4217-8e97-eb41da73f598,The Rolling Stones,7,Gimme Shelter,36228646-2c16-4a7a-9732-f4c14bb24776,1376381,9929777,6H3kDe7CGoWYBabAeVWGiD,1969-12-05,76,2024-09-12,6,0.733333,1960,True
2,69ee3720-a7cb-4402-b48d-a02c366f2bcf,The Cure,11,Friday I'm in Love,8d9104c6-7d8e-460b-8c3d-2d797f79953d,1804016,15974288,263aNAQCeFSWipk896byo6,1992-04-21,68,2024-09-12,10,8.733333,1990,True
3,83d91898-7763-47d7-b03b-b92132375c47,Pink Floyd,15,Wish You Were Here,feecff58-8ee2-4a7f-ac23-dc8ce7925286,1670937,17964458,6mFkJmJqdDVQ1REhVfGgd1,1975-09-12,74,2024-09-12,14,2.733333,1970,True
4,ada7a83c-e3e1-40f1-93f9-3e73dbc9298a,Arctic Monkeys,16,Do I Wanna Know?,f1e57531-e0df-4b3e-938f-1ae30c5b1a11,2210426,32317471,5FVd6KXrgO9B3JPmC8OPst,2013-09-09,86,2024-09-12,15,9.266667,2010,True
5,c8b03190-306c-4120-bb0b-6f2ebfc06ea9,The Weeknd,19,Blinding Lights,N/A,1781789,29564632,0VjIjW4GlUZAMYd2vXMi3b,2020-03-20,89,2024-09-12,18,12.266667,2020,True
6,f1cd52dc-49d1-4df6-93ea-3587d8fba30b,Charly García,20,Nos Siguen Pegando Abajo,4025e7dd-2a93-4d0c-a240-444c520da362,86746,845880,4VikOud5ZmdmHH6h7uQeDB,1983-11-05,66,2024-09-12,19,10.733333,1980,True
7,a74b1b7f-71a5-4011-9441-d0b5e4122711,Radiohead,1,Creep,d11fcceb-dfc5-4d19-b45d-f4e8f6d3eaa6,2976098,35502664,70LcF31zb1H0PyJoS1Sx1r,1993-02-22,87,2024-09-12,0,10.266667,1990,True
8,e21857d5-3256-4547-afb3-4b6ded592596,Gorillaz,14,Feel Good Inc.,5660558a-bfa7-416f-99d9-a34ca0a34515,2759699,28689243,0d28khcov6AiegSCpG5TuT,2005-05-23,84,2024-09-12,13,7.266667,2000,True
9,e6e1e76f-afee-4990-aad0-056199d94918,Babasónicos,17,Irresponsables,f2aecb3a-4344-4240-bc3e-befc0da657f6,104068,785423,0dsViRiDTIuexAL42Nc1Kh,2003-10-19,69,2024-09-12,16,7.733333,2000,True


## ORDENO COLUMNAS Y FILAS

In [11]:
if df_artists is not None:
  df_artists = df_artists[['FM_artist_mbid','FM_artist_name','FM_top_track_name','FM_top_track_mbid','FM_top_track_listeners','FM_top_track_playcount','SP_track_id','SP_track_popularity','SP_album_release_date','SP_album_decada','SP_track_popularity_media_distance']]
else:
    print("df_artists es None. Verifica la fuente de datos.")

## DELTALAKE SILVER

In [13]:
silver_dir = "datalake/silver"
raw_dir = f"/{silver_dir}/decads"
upsert_data_as_delta(df_artists,raw_dir,"target.SP_track_id = source.SP_track_id")